<a href="https://colab.research.google.com/github/Skk048/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Skk048/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

### Unit of Analysis
One row represents one pseudonymized content item. The dataset contains 30,000 rows across 44 columns and covers 32 clients.

### Time Window
The dataset contains trailing 90-day performance metrics. The reporting window should be interpreted according to the available date fields. Different clients may have different historical data coverage.

### Important Note
Each content item is the unit of analysis. Client IDs and content IDs are used for grouping and joining, not as model features.

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

### 1. Features
search_volume, competition, competition_level, cpc, content_type, main_intent, word_count, char_count, impressions_90d, clicks_90d, pageviews_90d, sessions_90d, users_90d, engaged_sessions_90d, ai_sessions_90d, scroll_events_90d, days_with_impressions, days_with_sessions, impressions_last_30d, clicks_last_30d, sessions_last_30d, impressions_prev_30d, clicks_prev_30d, sessions_prev_30d, content_age_days, days_since_last_update, freshness_tier, word_count_tier, char_count_tier, ctr, avg_position, engagement_rate, scroll_rate, ai_traffic_pct, impression_tier, position_tier.

These fields can be used as features only when they are available at prediction time.

### 2. Label
is_declining_label is the target label derived from trend_direction. It identifies whether content performance is declining.

### 3. Context
content_id, client_id, provider_used, model_used, age_tier and age_tier_order.

These fields are useful for identification, grouping, joining and analysis, but IDs must not be used as model features.

### 4. Excluded
trend_direction and trend_pct must be excluded from model features because they are used to derive the target label. Using them as features would cause target leakage.

In [ ]:
import pandas as pd

df = pd.read_csv('/content/content_refresh_anonymized.csv')

print("Total rows:", df.shape[0])
print("Total columns:", df.shape[1])
print("\nColumn names:")
print(df.columns.tolist())
for col in df.columns:
    print(col)


Total rows: 30000
Total columns: 44

Column names:
['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct']
content_id
client_id
search_volume
competition
competition_level
cpc
content_type
main_intent
word_count
char_count
provider_used
model_used
impressions_90d
clicks_90d
pageviews_

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

### Verification Results

1. **Grain:** The dataset contains 30,000 rows and 30,000 unique content IDs. No duplicate content IDs were found.

2. **Client Count:** There are 32 unique clients in the dataset.

3. **Missing Values:** The dataset contains missing values in several fields. Notably, trend_pct has 3,388 missing values, word_count has 7,699, and scroll_rate has 125.

4. **Category Pattern:** Missing word_count values are concentrated in the keyword_article content type (7,699 records).

5. **Ranking Data:** avg_position is zero in 1,205 records. According to the data dictionary, zero represents missing ranking data, not rank zero.

6. **Time Windows:** The dataset includes trailing 90-day metrics and separate last-30-day and previous-30-day metrics. These windows must be aligned before comparing or using them as features.

**Conclusion:** The dataset structure and major data-quality patterns were checked using Python queries. Missing values and time-window differences must be considered before analysis or modelling.

In [ ]:
# 1. Verify row count and grain
print("Total rows:", len(df))
print("Unique content IDs:", df['content_id'].nunique())
print("Duplicate IDs:", df['content_id'].duplicated().sum())

# 2. Verify client count
print("\nUnique clients:", df['client_id'].nunique())

# 3. Check missing values
print("\nMissing values:")
print(df.isnull().sum()[df.isnull().sum() > 0])

# 4. Missing values by content type
print("\nMissing word count by content type:")
print(df.groupby('content_type')['word_count'].apply(
    lambda x: x.isnull().sum()
))

# 5. Check ranking data
print("\nAverage position equal to zero:",
      (df['avg_position'] == 0).sum())

# 6. Check time windows
print("\nTime window columns:")
print([c for c in df.columns if
       '90d' in c or '30d' in c])

Total rows: 30000
Unique content IDs: 30000
Duplicate IDs: 0

Unique clients: 32

Missing values:
search_volume         2468
competition           2468
competition_level     2610
cpc                   2468
main_intent           2374
word_count            7699
char_count            7699
provider_used        21438
model_used            5733
word_count_tier       7699
char_count_tier       7699
scroll_rate            125
trend_pct             3388
dtype: int64

Missing word count by content type:
content_type
comparison article       0
feedly article           0
keyword article       7699
Name: word_count, dtype: int64

Average position equal to zero: 1205

Time window columns:
['impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d']


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

### Data Limitations

1. **Unbalanced History:** Different clients have different amounts of historical data. Therefore, comparisons should use client-specific time windows.

2. **Missing Data:** Missing values do not always mean zero activity. They may indicate unavailable or uncollected information.

3. **GSC Data Availability:** Early records may contain zero-filled Google Search Console metrics when GSC data is unavailable. The `gsc_data_available` flag must be checked.

4. **Ranking Limitations:** An average position of zero represents missing ranking information, not an actual ranking.

5. **Window Overlap:** The trailing 90-day metrics overlap with the last 30 days. These fields should not be combined without checking their time windows.

6. **Label Leakage:** `trend_direction` and `trend_pct` must not be used as model features because they are associated with the target label.

**Conclusion:** This dataset supports search-performance analysis, but it cannot guarantee future performance or prove causation. Results should be treated as decision-support evidence.

In [ ]:
# Check data limitations

# 1. Client distribution
print("Records per client:")
print(df['client_id'].value_counts().describe())

# 2. Content type distribution
print("\nContent type distribution:")
print(df['content_type'].value_counts())

# 3. Missing data pattern
print("\nMissing values in important columns:")
for col in ['word_count', 'scroll_rate', 'trend_pct']:
    print(col, ":", df[col].isnull().sum())

# 4. Check zero ranking values
print("\nZero average position:",
      (df['avg_position'] == 0).sum())

# 5. Check label leakage fields
print("\nLeakage-related fields:")
print(['trend_direction', 'trend_pct'])

# 6. Check overlapping time windows
print("\n90-day and 30-day metrics:")
print([col for col in df.columns if '90d' in col or '30d' in col])

Records per client:
count      32.000000
mean      937.500000
std      1376.387113
min         3.000000
25%       110.250000
50%       567.000000
75%      1058.750000
max      7008.000000
Name: count, dtype: float64

Content type distribution:
content_type
keyword article       27207
feedly article         2096
comparison article      697
Name: count, dtype: int64

Missing values in important columns:
word_count : 7699
scroll_rate : 125
trend_pct : 3388

Zero average position: 1205

Leakage-related fields:
['trend_direction', 'trend_pct']

90-day and 30-day metrics:
['impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d']


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.